# 03- Modelagem: prevendo a popularidade de jogos da Steam

**Pergunta:** dá para prever a faixa de `peak_ccu` (pico de jogadores simultâneos) a partir das características da página do jogo?

**Alvo principal:** classificação em 4 faixas (`0`, `1-20`, `21-500`, `500+`), com **F1-macro** como métrica, porque 77,7% dos jogos estão na faixa 0 e só 1,0% na faixa 500+ (a acurácia seria enganosa).
**Complemento:** regressão em `log1p(peak_ccu)`.

Foram escolhidos três modelos de classificação para o projeto:

- **Regressão Logística:** modelo utilizado como referência para classificação, com ajuste de pesos para lidar com o desbalanceamento das classes.
- **MLP:** rede neural com duas camadas ocultas, contendo 64 e 32 neurônios.
- **Random Forest:** conjunto de árvores de decisão, utilizando 200 árvores.

Também foi utilizado um **Baseline**, que sempre prevê a classe mais frequente. Ele serve como referência para verificar se os modelos conseguem superar uma previsão simples.


**Roteiro**
1. Carga dos dados e split estratificado
2. Pré-processamento
3. Tratamento do desbalanceamento
4. Validação cruzada (5 folds) no treino
5. Avaliação final no teste
6. Importância das variáveis
7. Regressão complementar
8. Discussão

> Antes de rodar: o 01 e o 02 precisam ter gerado `../data/processed/features.parquet`.

In [145]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.float_format", "{:.3f}".format)
SEED = 42

## 3.1 - Importação dos dados e split

In [146]:
dados = pd.read_parquet("../data/processed/features.parquet")
if "AppID" in dados.columns:
    dados = dados.set_index("AppID")

ordem = ["0", "1-20", "21-500", "500+"]

# alvos e colunas de vazamentos nunca entram como preditoras
alvos = [c for c in ["peak_ccu", "log_ccu", "faixa_ccu"] if c in dados.columns]
vazamentos = ["positive", "negative", "recommendations", "estimated_owners", "n_tags"]

print("faixa_ccu nulos:", dados["faixa_ccu"].isna().sum())
y_cls = dados["faixa_ccu"].astype(str)
y_reg = dados["log_ccu"] if "log_ccu" in dados.columns else np.log1p(dados["peak_ccu"])

x = dados.drop(columns=alvos + [c for c in vazamentos if c in dados.columns])
x = x.drop(columns=[c for c in x.columns if c.startswith(("pct_pos", "num_reviews"))])
x = x.select_dtypes(exclude=["datetime", "datetimetz"])

# categóricas viram códigos (NaN fica NaN); bool vira int8 só aqui, para o sklearn
for c in x.select_dtypes(["category", "object", "string"]).columns:
    x[c] = x[c].astype("category").cat.codes.replace(-1, np.nan)
x = x.astype({c: "int8" for c in x.select_dtypes("bool").columns})
# tipos anuláveis do pandas (Int64, boolean) viram float, que o sklearn aceita com NaN
for c in x.columns:
    if pd.api.types.is_extension_array_dtype(x[c]):
        x[c] = x[c].astype("float64")

print("x:", x.shape)
print("Colunas não numéricas restantes:", list(x.select_dtypes(exclude="number").columns))
print(y_cls.value_counts(normalize=True).reindex(ordem).round(3))

faixa_ccu nulos: 0
x: (83646, 42)
Colunas não numéricas restantes: []
faixa_ccu
0        0.777
1-20     0.173
21-500   0.041
500+     0.010
Name: proportion, dtype: float64


In [147]:
x_tr, x_te, y_tr, y_te, yr_tr, yr_te = train_test_split(
    x, y_cls, y_reg, test_size=0.2, stratify=y_cls, random_state=SEED
)
print("treino:", x_tr.shape, "| teste:", x_te.shape)
print(y_te.value_counts().reindex(ordem))

treino: (66916, 42) | teste: (16730, 42)
faixa_ccu
0         12994
1-20       2888
21-500      687
500+        161
Name: count, dtype: int64


## 3.2 - Pré-processamento

- **Regressão logística e MLP** precisam de escala: variáveis de cauda longa (`preco`, `dlc`, `conquistas`, `n_idiomas`...) passam por `log1p` e depois por padronização.
- **Random Forest** não precisa de escala nem de log, só de imputação.
- Valores faltantes (por exemplo, a nota do Metacritic) são preenchidos com a mediana; a flag `tem_metacritic` já guarda a informação de que o dado não existia.
- Os pré-processadores ficam dentro do `Pipeline`, então são ajustados só no treino de cada fold (sem vazamento da validação).

In [148]:
cauda_longa = ["preco", "dlc", "conquistas", "n_idiomas", "n_categorias", "n_generos", "n_devs", "idade_minima"]
cols_log = [c for c in cauda_longa if c in X.columns]
cols_resto = [c for c in X.columns if c not in cols_log]


def prep_linear():
    return ColumnTransformer([
        ("log", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("log", FunctionTransformer(np.log1p)),
            ("esc", StandardScaler()),
        ]), cols_log),
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("esc", StandardScaler()),
        ]), cols_resto),
    ])


def prep_arvore():
    return ColumnTransformer([("num", SimpleImputer(strategy="median"), list(X.columns))])

## 3.3 - Balanceamento das classes no treino

- **Regressão logística e Random Forest:** `class_weight` (`balanced` e `balanced_subsample`).
- **MLP:** o `MLPClassifier` do scikit-learn não tem `class_weight`. Solução: reamostrar **só o treino** de cada fold, com `N_POR_CLASSE` linhas por classe (subamostra a classe 0, que é enorme, e repete linhas das classes raras). A validação e o teste ficam com a distribuição original.

Limitação: na faixa `500+` o treino tem poucas centenas de jogos, então a repetição de linhas aumenta o risco de a rede decorar esses exemplos.

In [149]:
np_classe = 11000


def rebalancear(X_, y_, n=np_classe, seed=SEED):
    rng = np.random.default_rng(seed)
    idx = []
    for classe in np.unique(y_):
        pos = np.flatnonzero(y_.values == classe)
        idx.append(rng.choice(pos, size=n, replace=len(pos) < n))
    idx = np.concatenate(idx)
    rng.shuffle(idx)
    return X_.iloc[idx], y_.iloc[idx]

## 3.4 - Modelos e validação cruzada 

In [150]:
def logreg():
    return Pipeline([
        ("prep", prep_linear()),
        ("m", LogisticRegression(
            class_weight="balanced", 
            max_iter=1000,
            random_state=SEED)),
    
    ])


def mlp():
    return Pipeline([
        ("prep", prep_linear()),
        ("m", MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-3, max_iter=100, random_state=SEED)),
    ])


def random_forest():
    return Pipeline([
        ("prep", prep_arvore()),
        ("m", RandomForestClassifier(
            n_estimators=200, min_samples_leaf=2, 
            class_weight="balanced_subsample",
            n_jobs=-1, random_state=SEED)),
    ])


modelos = {
    "Baseline (classe majoritária)": (lambda: DummyClassifier(strategy="most_frequent"), False),
    "Regressão logística": (logreg, False),
    "MLP": (mlp, True),
    "Random Forest": (random_forest, False),
}

### Para avaliar os modelos, foram utilizadas diferentes métricas:

- **F1-score:** combina precisão e recall em uma única medida. Quanto maior o resultado, melhor o equilíbrio entre essas duas métricas.
- **F1-macro:** calcula o F1-score de cada classe e faz uma média simples entre elas. Dessa forma, todas as classes possuem o mesmo peso, independentemente da quantidade de jogos.
- **Acurácia balanceada:** calcula a média do recall de todas as classes, evitando que as classes mais numerosas dominem o resultado.

In [151]:
import os
import time
import warnings
from joblib import Parallel, delayed

n_par = min(5, os.cpu_count() or 1)
folds_em_paralelo = {"Regressão logística": n_par, "MLP": n_par}


def _um_fold(construir, X_, y_, tr, va, reb):
    warnings.filterwarnings("ignore", category=ConvergenceWarning)  # os processos novos não herdam o filtro
    Xf, yf = X_.iloc[tr], y_.iloc[tr]
    if reb:
        Xf, yf = rebalancear(Xf, yf)
    modelo = construir().fit(Xf, yf)
    return f1_score(y_.iloc[va], modelo.predict(X_.iloc[va]), average="macro")


def avaliar_cv(construir, X_, y_, com_rebalanceamento=False, n_splits=5, n_jobs=1):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    notas = Parallel(n_jobs=n_jobs)(
        delayed(_um_fold)(construir, X_, y_, tr, va, com_rebalanceamento)
        for tr, va in skf.split(X_, y_)
    )
    return np.array(notas)


linhas = []
for nome, (construir, reb) in modelos.items():
    print(f"{nome:32s}", end=" ", flush=True)
    t0 = time.time()
    notas = avaliar_cv(construir, X_tr, y_tr, reb, n_jobs=folds_em_paralelo.get(nome, 1))
    linhas.append({"modelo": nome, "F1-macro (média)": notas.mean(), "desvio": notas.std()})
    print(f"{notas.mean():.3f} ± {notas.std():.3f}  ({time.time() - t0:.0f}s)")

tabela_cv = pd.DataFrame(linhas).set_index("modelo")
tabela_cv

Baseline (classe majoritária)    0.219 ± 0.000  (1s)
Regressão logística              0.283 ± 0.003  (5s)
MLP                              0.282 ± 0.004  (43s)
Random Forest                    0.301 ± 0.004  (25s)


,F1-macro (média),desvio
modelo,,
Baseline (classe majoritária),0.219,0.000
Regressão logística,0.283,0.003
MLP,0.282,0.004
Random Forest,0.301,0.004


#### Validação cruzada (Cross-validation)

Foi utilizada a validação cruzada com 5 folds. Os folds são divisões do conjunto de treinamento em cinco partes.

Em cada repetição, quatro partes são utilizadas para treinar o modelo e uma para validação. Esse processo é repetido cinco vezes, alternando a parte utilizada para validação.

Ao final, são calculadas a média e o desvio dos resultados. A média representa o desempenho geral, enquanto o desvio indica quanto os resultados variaram entre as divisões.